# Lab 3: Extracting Structured Tables and Financial Data (with Explainability)

### Install Required Libraries

In [ ]:
# Install the required libraries
!pip install pageindex langchain-aws boto3 requests

  Using cached groq-0.37.1-py3-none-any.whl.metadata (16 kB)
Using cached groq-0.37.1-py3-none-any.whl (137 kB)
  Attempting uninstall: groq
    Found existing installation: groq 1.5.0
    Uninstalling groq-1.5.0:
      Successfully uninstalled groq-1.5.0



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Import Libraries

In [ ]:
# Core modules
import os
import time
import re
import requests

# PageIndex client for vectorless retrieval
from pageindex import PageIndexClient
import pageindex.utils as utils

# LangChain's AWS Bedrock wrapper
from langchain_aws import ChatBedrockConverse

### Setup API Keys

In [3]:
# --- Configure AWS Bedrock credentials ---
os.environ["AWS_ACCESS_KEY_ID"]     = "YOUR_ACCESS_KEY_ID"
os.environ["AWS_SECRET_ACCESS_KEY"] = "YOUR_SECRET_ACCESS_KEY"
os.environ["AWS_ENDPOINT_URL"]      = "https://api.enterprisesi.co/api/v1/aws-genai/bedrock-runtime"
os.environ["AWS_REGION"]            = "ap-south-1"

print("Credentials configured.")

# --- Load PageIndex API Key ---
PAGEINDEX_API_KEY = input("Enter your PageIndex API key (get one at https://pageindex.ai): ").strip()
os.environ["PAGEINDEX_API_KEY"] = PAGEINDEX_API_KEY

print("PageIndex key loaded.")

# Initialize the PageIndex Client
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

### Download the Financial Document from an Online Link

In [ ]:
import os

# Paste the direct link to the Earnings Release PDF you want to analyze
PDF_URL = "https://s21.q4cdn.com/736796105/files/doc_financials/2025/q4/Exhibit-99-1-Q4-2025-Earnings-Release.pdf"

# Save the PDF locally so it can be handed to PageIndex
os.makedirs("data", exist_ok=True)
PDF_PATH = os.path.join("data", PDF_URL.split("/")[-1])

response = requests.get(PDF_URL)
response.raise_for_status()
with open(PDF_PATH, "wb") as f:
    f.write(response.content)

print(f"Downloaded the financial document to '{PDF_PATH}'")

Success: Found the financial document at 'data/CCS 3.31.25 Earnings Release 8-K Exhibit 99.1.pdf'


### Index the Document (Preserving Tables)

In [ ]:
# Submit the document — PageIndex keeps tables whole as single nodes
doc_info = pi_client.submit_document(PDF_PATH)
doc_id = doc_info["doc_id"]

print(f"Document Submitted. Tracking ID: {doc_id}")

print("Waiting for the document to be indexed...")
while not pi_client.is_retrieval_ready(doc_id):
    print("Mapping tables and sections... checking again in 5 seconds.")
    time.sleep(5)

print("Indexing Complete! The document's structure is fully preserved.")

Document Submitted. Tracking ID: pi-cmrn6rf4j01fx01o42s2azgoc
Waiting for the document to be indexed...
Mapping tables and sections... checking again in 5 seconds.
Mapping tables and sections... checking again in 5 seconds.
Mapping tables and sections... checking again in 5 seconds.
Indexing Complete! The document's structure is fully preserved.


In [14]:
tree = pi_client.get_tree(doc_id, node_summary=True)["result"]
print("Document Tree Structure:")
utils.print_tree(tree)

Document Tree Structure:
[{'title': 'Century Communities Reports First Quarte...',
  'node_id': '0000',
  'summary': 'Century Communities reported its Q1 2025...'},
 {'title': 'First Quarter 2025 Results',
  'node_id': '0001',
  'prefix_summary': 'The report details the Q1 2025 financial...',
  'nodes': [{'title': 'Balance Sheet and Liquidity',
             'node_id': '0002',
             'summary': 'In Q1 2025, the company maintained a str...'},
            {'title': 'Full Year 2025 Outlook',
             'node_id': '0003',
             'summary': '## Full Year 2025 Outlook\n\nScott Dixon, ...'},
            {'title': 'Webcast and Conference Call',
             'node_id': '0004',
             'summary': 'Century Communities will host a webcast ...'},
            {'title': 'About Century Communities',
             'node_id': '0005',
             'summary': 'Century Communities, Inc. is a prominent...'},
            {'title': 'Non-GAAP Financial Measures',
             'node_id': '0006'

### Initialize the LLM

In [ ]:
# Set up the LLM
llm = ChatBedrockConverse(
    model="global.amazon.nova-2-lite-v1:0",
    temperature=0.1  # Kept low so answers stay precise, not creative
)

### Define Table-Safe Retrieval Function (Tagged for Explainability)

In [ ]:
def retrieve_from_pageindex(query, doc_id, top_k=2):
    """
    Retrieves whole logical sections (like full tables) that match the query.
    Every table returned here is tagged with:
      - which table it was (1st match, 2nd match, ...)
      - the section/table title
      - the node id (the tree's unique ID for that section)
      - the page number(s) the table came from
    This metadata is what lets us explain WHY a table was picked, later.
    """
    response = pi_client.submit_query(doc_id=doc_id, query=query)
    retrieval_id = response.get("retrieval_id")

    if not retrieval_id:
        return []

    # Poll until the search finishes
    while True:
        retrieval = pi_client.get_retrieval(retrieval_id)
        status = retrieval.get("status")
        if status == "completed":
            break
        elif status == "failed":
            return []
        time.sleep(1)

    nodes = retrieval.get("retrieved_nodes", [])
    tables = []

    # Extract the full tables/sections, one at a time
    for index, node in enumerate(nodes[:top_k]):
        node_name = node.get("title") or f"Table {index + 1}"
        node_id = node.get("id", "unknown")  # PageIndex returns the node's ID under "id"
        relevant_contents = node.get("relevant_contents", [])

        section_text = []
        page_numbers = []
        for group in relevant_contents:
            for item in group:
                content = item.get("relevant_content")
                if content:
                    section_text.append(content)

                # page number is embedded in a string like "<physical_index_6>"
                raw_page = item.get("physical_index", "")
                match = re.search(r"(\d+)", raw_page) if isinstance(raw_page, str) else None
                if match:
                    page_num = int(match.group(1))
                    if page_num not in page_numbers:
                        page_numbers.append(page_num)

        tables.append({
            "table_number": index + 1,
            "section": node_name,
            "node_id": node_id,
            "pages": page_numbers,
            "text": "\n".join(section_text)
        })

    return tables

### Build Vectorless RAG Pipeline (With Table Citations)

In [ ]:
def vectorless_rag(query, doc_id):
    # Retrieve tables tagged with their number, node id, and pages
    tables = retrieve_from_pageindex(query, doc_id)

    if not tables:
        return "No relevant context found.", [], ""

    # Label each table clearly so the LLM can refer back to it (e.g. "[Table 1]")
    labeled_context = "\n\n".join(
        f"[Table {t['table_number']} - {t['section']}]\n{t['text']}" for t in tables
    )

    # Ask the LLM to answer using only the tables, citing each value's table
    prompt = f"""
You are a data analyst. Answer the question ONLY using the provided text/tables below.
Pay strict attention to table rows, columns, and footnotes. Do not round or approximate values unless asked.

CRITICAL INSTRUCTIONS:
- Every value you use MUST be tagged with its table, like this: [Table 1]
- If you use values from more than one table, tag each one separately.
- If the data is not in the text, say "Not found in document."

Context:
{labeled_context}

Question: {query}

Be concise in your answer.
"""

    response = llm.invoke(prompt)
    final_answer = response.content

    return final_answer, tables, labeled_context

### Query a Specific Table Metric

In [ ]:
# Let's ask a question that requires the LLM to look at a specific row and column in a financial table.
# (You can change this query based on the exact contents of your Earnings Release PDF)
query = "Looking at the Commodity Data within the Summary of Rail Data, what was the total freight revenue for Automotive for the year ended December 31, 2025, compared to 2024, and what was the percentage change?"

In [ ]:
print(f"Question: {query}\n")
print("Locating the correct financial table...\n")

# Run the pipeline
final_answer, tables, labeled_context = vectorless_rag(query, doc_id)

# Show which tables were searched
print("--- TABLES SEARCHED ---")
for t in tables:
    print(f"Table {t['table_number']}: {t['section']}")

print("\n--- FINAL ANSWER ---")
print(final_answer)

Question: What was the total revenue for the three months ended March 31, 2025?

Locating the correct financial table...

--- TABLES SEARCHED ---
Table 1: Forward-Looking Statements

--- FINAL ANSWER ---
The total revenue for the three months ended March 31, 2025, was $903,232 [Table 1].


In [ ]:
print("\n--- EXPLAINABILITY ---")
for t in tables:
    pages = ", ".join(str(p) for p in t["pages"]) if t["pages"] else "unknown"

    # Check if the citation tag is actually present in the LLM's answer
    was_used = f"[Table {t['table_number']}]" in final_answer

    status = "USED in answer" if was_used else "retrieved but NOT used"

    print(f"\nTable {t['table_number']}: \"{t['section']}\"")
    print(f"node_id: {t['node_id']} | page(s): {pages} | {status}")

    # Ask the LLM to explain why this table is relevant
    explain_prompt = f"""
In 3-4 short lines, explain why the table below is relevant to the question.
Be specific -- mention the actual numbers or rows in the table that connect to the question.
Do not repeat the question. Do not add extra commentary.

Question: {query}

Table title: {t['section']}
Table content: {t['text']}
"""
    explanation_response = llm.invoke(explain_prompt)
    print(f"Why: {explanation_response.content.strip()}")


--- EXPLAINABILITY ---

Table 1: "Forward-Looking Statements"
node_id: 0007 | page(s): 4 | USED in answer
Why: The table is relevant to the question because it contains the total revenue for the three months ended March 31, 2025, which is $903,232.
